# Tutorial 06: Implementing Custom Algorithms

Learn how to implement your own heuristic algorithms for solving VRP problems.

**What you'll learn:**
- Understand the Solver interface
- Implement a simple greedy heuristic from scratch
- Create a local search improvement algorithm
- Extend existing algorithms with custom operators
- Compare your algorithm with ALNS

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Tutorial 03 (Custom Problems)
- Basic understanding of heuristic algorithms

**Time:** ~50 minutes

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import random
from typing import List, Tuple
import time
import matplotlib.pyplot as plt

# VRP Toolkit imports
from vrp_toolkit.problems.pdptw import PDPTWInstance, Node
from vrp_toolkit.algorithms.base import Solver, VRPProblem, VRPSolution
from vrp_toolkit.algorithms.alns.solver import ALNSSolver
from vrp_toolkit.data.generators import OrderGenerator

# Verify imports
print("All imports successful!")

## 2. Quick Start: Simplest Custom Solver

Let's create the **simplest possible solver**: a random assignment algorithm that randomly assigns customers to vehicles.

In [ ]:
class RandomSolver(Solver):
    """Randomly assign customers to routes."""
    
    def solve(self, problem: VRPProblem, num_vehicles: int = 2) -> VRPSolution:
        """Create random solution."""
        # Get pickup-delivery pairs
        n = problem.n
        pairs = [(i, i + n) for i in range(1, n + 1)]
        
        # Shuffle and split into routes
        random.shuffle(pairs)
        routes_per_vehicle = len(pairs) // num_vehicles
        
        routes = []
        for v in range(num_vehicles):
            start = v * routes_per_vehicle
            end = start + routes_per_vehicle if v < num_vehicles - 1 else len(pairs)
            
            # Build route: depot -> pickups -> deliveries -> depot
            route = [0]
            for pickup, delivery in pairs[start:end]:
                route.extend([pickup, delivery])
            route.append(0)
            routes.append(route)
        
        return problem.create_solution(routes)

# Test it
generator = OrderGenerator(num_orders=4, num_vehicles=2)
instance = generator.generate_instance()

random_solver = RandomSolver()
random_solution = random_solver.solve(instance)

print(f"Random solution created!")
print(f"Routes: {random_solution.routes}")
print(f"Cost: {random_solution.objective_value():.2f}")
print(f"Feasible: {random_solution.is_feasible()}")

**What just happened:**
- We created a custom solver by inheriting from `Solver` base class
- Implemented the `solve(problem, **kwargs) -> VRPSolution` method
- Solution might be infeasible (violates constraints), but it's fast!
- This is our baseline - any real algorithm should beat this

## 3. Understanding the Solver Interface

### 3.1 The Solver Base Class

All solvers must implement:
```python
class MySolver(Solver):
    def solve(self, problem: VRPProblem, **kwargs) -> VRPSolution:
        # Your algorithm here
        routes = [...]
        return problem.create_solution(routes)
```

**Key methods:**
- `solve(problem, **kwargs)` - Main solving method (REQUIRED)
- `problem.create_solution(routes)` - Convert routes to VRPSolution
- `solution.objective_value()` - Get solution cost
- `solution.is_feasible()` - Check constraint satisfaction

### 3.2 Accessing Problem Data

The `problem` object provides all instance data:

In [ ]:
# Example: Accessing problem data
print("Problem data available:")
print(f"  Number of pickup-delivery pairs: {instance.n}")
print(f"  Total nodes: {len(instance.nodes)}")
print(f"  Distance matrix shape: {instance.distance_matrix.shape}")
print(f"  Vehicle capacity: {instance.vehicle_capacity}")
print(f"  Battery capacity: {instance.battery_capacity}")
print(f"  Pickup-delivery pairs: {instance.pickup_delivery_pairs}")

# Access specific node
node_1 = instance.nodes[1]
print(f"\nNode 1 details:")
print(f"  Type: {node_1.node_type}")
print(f"  Location: ({node_1.x}, {node_1.y})")
print(f"  Time window: {node_1.time_window}")
print(f"  Demand: {node_1.demand}")

### 3.3 Creating and Evaluating Solutions

**Routes format:** List of lists, each inner list is a route starting and ending at depot (node 0)

In [ ]:
# Example: Manual solution creation
manual_routes = [
    [0, 1, 2, 3, 4, 0],  # Route 1: depot -> pickup 1 -> delivery 1 -> pickup 2 -> delivery 2 -> depot
    [0, 5, 6, 7, 8, 0]   # Route 2: depot -> pickup 3 -> delivery 3 -> pickup 4 -> delivery 4 -> depot
]

manual_solution = instance.create_solution(manual_routes)

print("Manual solution:")
print(f"  Cost: {manual_solution.objective_value():.2f}")
print(f"  Feasible: {manual_solution.is_feasible()}")
print(f"  Routes: {manual_solution.routes}")

## 4. Implementing Heuristic Algorithms

### 4.1 Nearest Neighbor Heuristic

**Algorithm:** Always visit the nearest unvisited customer.

**Steps:**
1. Start at depot
2. Find nearest unvisited pickup
3. Visit pickup, then its delivery
4. Repeat until all customers visited
5. Return to depot

In [ ]:
class NearestNeighborSolver(Solver):
    """Nearest neighbor construction heuristic."""
    
    def solve(self, problem: VRPProblem, num_vehicles: int = 1) -> VRPSolution:
        """Build solution by always choosing nearest unvisited customer."""
        n = problem.n
        dist_matrix = problem.distance_matrix
        
        # Track unvisited pickups
        unvisited_pickups = set(range(1, n + 1))
        
        routes = []
        
        for _ in range(num_vehicles):
            if not unvisited_pickups:
                break
            
            route = [0]  # Start at depot
            current = 0
            
            # Build route
            while unvisited_pickups:
                # Find nearest unvisited pickup
                nearest = min(unvisited_pickups, key=lambda p: dist_matrix[current, p])
                
                # Visit pickup then delivery
                pickup = nearest
                delivery = pickup + n
                
                route.append(pickup)
                route.append(delivery)
                
                unvisited_pickups.remove(pickup)
                current = delivery
                
                # Simple capacity check (optional: can break route early)
                if len(unvisited_pickups) > 0 and len(route) > 10:  # Arbitrary limit
                    break
            
            route.append(0)  # Return to depot
            routes.append(route)
        
        return problem.create_solution(routes)

# Test nearest neighbor
nn_solver = NearestNeighborSolver()
nn_solution = nn_solver.solve(instance, num_vehicles=2)

print("Nearest Neighbor Solution:")
print(f"  Routes: {nn_solution.routes}")
print(f"  Cost: {nn_solution.objective_value():.2f}")
print(f"  Feasible: {nn_solution.is_feasible()}")

### 4.2 Savings Algorithm

**Algorithm:** Merge routes based on savings from combining them.

**Savings formula:** `s(i,j) = d(0,i) + d(0,j) - d(i,j)`

**Steps:**
1. Start with each customer in separate route
2. Calculate savings for all pairs
3. Sort savings in descending order
4. Merge routes with highest savings (if feasible)
5. Repeat until no more merges possible

In [ ]:
class SavingsSolver(Solver):
    """Clarke-Wright Savings Algorithm."""
    
    def solve(self, problem: VRPProblem, **kwargs) -> VRPSolution:
        """Build solution using savings algorithm."""
        n = problem.n
        dist_matrix = problem.distance_matrix
        
        # Step 1: Initialize - each customer in separate route
        routes = []
        for i in range(1, n + 1):
            pickup = i
            delivery = i + n
            routes.append([0, pickup, delivery, 0])
        
        # Step 2: Calculate savings for all pairs
        savings = []
        for i in range(1, n + 1):
            for j in range(i + 1, n + 1):
                # Savings from merging route ending at delivery_i with route starting at pickup_j
                delivery_i = i + n
                pickup_j = j
                s = dist_matrix[delivery_i, 0] + dist_matrix[0, pickup_j] - dist_matrix[delivery_i, pickup_j]
                savings.append((s, i, j))
        
        # Step 3: Sort savings in descending order
        savings.sort(reverse=True)
        
        # Step 4: Merge routes with highest savings
        for s, i, j in savings:
            # Find routes containing i and j
            route_i = None
            route_j = None
            
            for route in routes:
                if i in route or i + n in route:
                    route_i = route
                if j in route or j + n in route:
                    route_j = route
            
            # Can merge if routes are different and compatible
            if route_i != route_j and route_i is not None and route_j is not None:
                # Check if i is at end of route_i and j is at start of route_j
                if route_i[-2] == i + n and route_j[1] == j:
                    # Merge: route_i + route_j (remove depot in middle)
                    merged = route_i[:-1] + route_j[1:]
                    routes.remove(route_i)
                    routes.remove(route_j)
                    routes.append(merged)
        
        return problem.create_solution(routes)

# Test savings algorithm
savings_solver = SavingsSolver()
savings_solution = savings_solver.solve(instance)

print("Savings Algorithm Solution:")
print(f"  Number of routes: {len(savings_solution.routes)}")
print(f"  Cost: {savings_solution.objective_value():.2f}")
print(f"  Feasible: {savings_solution.is_feasible()}")

### 4.3 Local Search Improvement

**Algorithm:** Improve an existing solution using 2-opt local search.

**2-opt:** Remove two edges and reconnect in different way.

In [ ]:
class TwoOptSolver(Solver):
    """2-opt local search improvement."""
    
    def __init__(self, initial_solver: Solver = None, max_iterations: int = 100):
        self.initial_solver = initial_solver or NearestNeighborSolver()
        self.max_iterations = max_iterations
    
    def solve(self, problem: VRPProblem, **kwargs) -> VRPSolution:
        """Improve initial solution with 2-opt."""
        # Get initial solution
        solution = self.initial_solver.solve(problem, **kwargs)
        dist_matrix = problem.distance_matrix
        
        improved = True
        iteration = 0
        
        while improved and iteration < self.max_iterations:
            improved = False
            iteration += 1
            
            # Try to improve each route
            for route_idx, route in enumerate(solution.routes):
                if len(route) <= 4:  # Too short for 2-opt
                    continue
                
                # Try all 2-opt swaps
                for i in range(1, len(route) - 2):
                    for j in range(i + 1, len(route) - 1):
                        # Current edges: (route[i-1], route[i]) and (route[j], route[j+1])
                        # New edges: (route[i-1], route[j]) and (route[i], route[j+1])
                        
                        old_dist = (dist_matrix[route[i-1], route[i]] + 
                                    dist_matrix[route[j], route[j+1]])
                        new_dist = (dist_matrix[route[i-1], route[j]] + 
                                    dist_matrix[route[i], route[j+1]])
                        
                        if new_dist < old_dist:
                            # Perform 2-opt swap: reverse segment between i and j
                            solution.routes[route_idx] = (route[:i] + 
                                                          route[i:j+1][::-1] + 
                                                          route[j+1:])
                            improved = True
                            break
                    if improved:
                        break
        
        return problem.create_solution(solution.routes)

# Test 2-opt improvement
twoopt_solver = TwoOptSolver(initial_solver=NearestNeighborSolver(), max_iterations=50)
twoopt_solution = twoopt_solver.solve(instance, num_vehicles=2)

print("2-Opt Improved Solution:")
print(f"  Cost: {twoopt_solution.objective_value():.2f}")
print(f"  vs Nearest Neighbor: {nn_solution.objective_value():.2f}")
print(f"  Improvement: {((nn_solution.objective_value() - twoopt_solution.objective_value()) / nn_solution.objective_value() * 100):.1f}%")

## 5. Advanced: Custom ALNS Operators

You can extend ALNS with custom destruction/repair operators.

### 5.1 Custom Removal Operator

**Example:** Remove customers clustered together (cluster-based removal)

In [ ]:
def cluster_removal(solution, instance, num_remove=3):
    """
    Remove spatially clustered customers.
    
    Args:
        solution: Current solution
        instance: Problem instance
        num_remove: Number of customers to remove
    
    Returns:
        Modified solution with customers removed
    """
    # Pick random seed customer
    all_pickups = [i for i in range(1, instance.n + 1)]
    if not all_pickups:
        return solution
    
    seed = random.choice(all_pickups)
    
    # Find nearest customers to seed
    distances = [(i, instance.distance_matrix[seed, i]) for i in all_pickups if i != seed]
    distances.sort(key=lambda x: x[1])
    
    # Remove seed + nearest neighbors
    to_remove = [seed] + [i for i, d in distances[:num_remove-1]]
    
    # Remove from routes
    new_routes = []
    for route in solution.routes:
        new_route = [node for node in route if node not in to_remove and node - instance.n not in to_remove]
        if len(new_route) > 2:  # Keep route if not empty
            new_routes.append(new_route)
    
    return instance.create_solution(new_routes)

# Test cluster removal
test_solution = nn_solution
print(f"Original cost: {test_solution.objective_value():.2f}")
removed_solution = cluster_removal(test_solution, instance, num_remove=2)
print(f"After cluster removal: {removed_solution.objective_value():.2f}")
print(f"Routes: {removed_solution.routes}")

### 5.2 Custom Repair Operator

**Example:** Insert customers at positions minimizing detour cost

In [ ]:
def cheapest_insertion_repair(solution, instance, unvisited):
    """
    Repair solution by inserting unvisited customers at cheapest position.
    
    Args:
        solution: Partial solution
        instance: Problem instance
        unvisited: List of unvisited pickup nodes
    
    Returns:
        Complete solution
    """
    dist_matrix = instance.distance_matrix
    n = instance.n
    
    routes = [route.copy() for route in solution.routes]
    
    # Insert each unvisited customer
    for pickup in unvisited:
        delivery = pickup + n
        
        best_cost = float('inf')
        best_route_idx = 0
        best_pickup_pos = 1
        best_delivery_pos = 2
        
        # Try inserting into each route
        for route_idx, route in enumerate(routes):
            # Try all positions for pickup and delivery
            for p_pos in range(1, len(route)):
                for d_pos in range(p_pos + 1, len(route) + 1):
                    # Calculate insertion cost
                    cost = 0
                    
                    # Cost of inserting pickup
                    cost += dist_matrix[route[p_pos-1], pickup]
                    cost += dist_matrix[pickup, route[p_pos] if p_pos < len(route) else 0]
                    cost -= dist_matrix[route[p_pos-1], route[p_pos] if p_pos < len(route) else 0]
                    
                    # Cost of inserting delivery
                    d_pos_adj = d_pos if p_pos >= d_pos else d_pos + 1
                    if d_pos_adj <= len(route):
                        cost += dist_matrix[route[d_pos_adj-1] if d_pos_adj > 0 else 0, delivery]
                        cost += dist_matrix[delivery, route[d_pos_adj] if d_pos_adj < len(route) else 0]
                        cost -= dist_matrix[route[d_pos_adj-1] if d_pos_adj > 0 else 0, 
                                            route[d_pos_adj] if d_pos_adj < len(route) else 0]
                    
                    if cost < best_cost:
                        best_cost = cost
                        best_route_idx = route_idx
                        best_pickup_pos = p_pos
                        best_delivery_pos = d_pos
        
        # Insert at best position
        routes[best_route_idx].insert(best_pickup_pos, pickup)
        routes[best_route_idx].insert(best_delivery_pos, delivery)
    
    return instance.create_solution(routes)

print("Custom operators created successfully!")

## 6. Real-World Example: Algorithm Comparison

Let's compare all our algorithms on a realistic instance.

In [ ]:
# Create test instance
test_generator = OrderGenerator(num_orders=10, num_vehicles=3)
test_instance = test_generator.generate_instance()

print("Testing all algorithms on 10-order instance...\n")

# Benchmark all solvers
solvers = [
    ('Random', RandomSolver()),
    ('Nearest Neighbor', NearestNeighborSolver()),
    ('Savings', SavingsSolver()),
    ('2-Opt (NN)', TwoOptSolver(NearestNeighborSolver())),
    ('ALNS', ALNSSolver())
]

results = []

for name, solver in solvers:
    start = time.time()
    solution = solver.solve(test_instance, num_vehicles=3)
    elapsed = time.time() - start
    
    results.append({
        'Algorithm': name,
        'Cost': solution.objective_value(),
        'Feasible': solution.is_feasible(),
        'Routes': len(solution.routes),
        'Time (s)': elapsed
    })

# Display results
import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

# Find best
best = min(results, key=lambda x: x['Cost'])
print(f"\nBest algorithm: {best['Algorithm']} (cost: {best['Cost']:.2f})")

In [ ]:
# Visualize comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cost comparison
algorithms = [r['Algorithm'] for r in results]
costs = [r['Cost'] for r in results]
colors = ['red' if not r['Feasible'] else 'green' for r in results]

ax1.bar(algorithms, costs, color=colors, alpha=0.7)
ax1.set_ylabel('Solution Cost')
ax1.set_title('Algorithm Cost Comparison')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Time comparison
times = [r['Time (s)'] for r in results]
ax2.bar(algorithms, times, color='blue', alpha=0.7)
ax2.set_ylabel('Runtime (seconds)')
ax2.set_title('Algorithm Runtime Comparison')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Green bars: Feasible solutions")
print("- Red bars: Infeasible solutions (violate constraints)")
print("- Tradeoff: Better solutions often need more time")

## 7. Best Practices for Custom Algorithms

**When implementing custom solvers:**

1. **Start simple:** Get a basic version working first
2. **Test feasibility:** Check `solution.is_feasible()` frequently
3. **Use existing solvers:** Combine/extend rather than reinvent
4. **Profile performance:** Measure time for large instances
5. **Handle edge cases:** Empty routes, single customer, etc.

**Common pitfalls:**
- **Forgetting depot:** Routes must start and end at node 0
- **Pickup before delivery:** Always visit pickup before paired delivery
- **Index errors:** Remember delivery node = pickup node + n
- **Distance matrix access:** Use `dist_matrix[i, j]` not `dist_matrix[i][j]`
- **Infinite loops:** Add iteration limits to while loops

**Debugging tips:**
- Print routes at each step
- Visualize solutions with `solution.plot()`
- Start with tiny instances (2-3 customers)
- Check intermediate solutions for feasibility

## 8. Practice Exercises

1. **Basic:** Implement a "Farthest Insertion" heuristic that always inserts the customer farthest from the current route.

2. **Intermediate:** Modify the 2-Opt solver to also try 3-opt moves (removing 3 edges and reconnecting).

3. **Advanced:** Implement a simple Genetic Algorithm solver with population, crossover, and mutation operators.

**Hints:**
- For Exercise 1: Similar to Nearest Neighbor but use max() instead of min()
- For Exercise 2: 3-opt has 7 different reconnection ways
- For Exercise 3: Start with population of random solutions, evolve over generations

In [ ]:
# Your solutions here


## 9. Summary

**What you learned:**
- ✅ Implement custom solvers using the Solver interface
- ✅ Create construction heuristics (Nearest Neighbor, Savings)
- ✅ Implement improvement algorithms (2-Opt)
- ✅ Extend ALNS with custom operators
- ✅ Benchmark and compare algorithms

**Key takeaways:**
1. All solvers implement `solve(problem, **kwargs) -> VRPSolution`
2. Construction heuristics build solutions from scratch
3. Improvement algorithms refine existing solutions
4. Hybrid approaches often work best (construct + improve)
5. Simple heuristics can be surprisingly effective

**Next steps:**
- Explore **Tutorial 07: Data Generation** for testing algorithms
- Study ALNS source code for advanced metaheuristic patterns
- Implement algorithms from recent research papers
- Benchmark on standard datasets (Solomon, Li & Lim)